# DB Query Template

`.env.development.local`의 개발 DB 환경변수를 사용해서 MySQL 조회 쿼리를 실행합니다.

In [ ]:
# 처음 한 번만 필요하면 주석을 풀고 실행하세요.
# %pip install pandas sqlalchemy pymysql python-dotenv

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   - -------------------------------------- 0.1/2.1 MB 1.7 MB/s eta 0:00:02
   ---------------------------------------- 2.1/2.1 MB 34.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/45.7 kB ? eta -:--:--
   ---------------------------------------- 45.7/45.7 kB ? eta 0:00:00
   ---------------------------------------- 0.0/239.1 kB ? eta -:--:--
   ---------------------------------------- 239.1/239.1 kB 4.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os
from pathlib import Path
from urllib.parse import quote_plus

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv(Path('.env.development.local'))

db_host = os.getenv('DB_HOST')
db_port = os.getenv('DB_PORT', '3306')
db_name = os.getenv('DB_NAME')
db_user = os.getenv('DB_USER')
db_password = os.getenv('DB_PASSWORD')

database_url = (
    f"mysql+pymysql://{quote_plus(db_user)}:{quote_plus(db_password)}"
    f"@{db_host}:{db_port}/{db_name}?charset=utf8mb4"
)

engine = create_engine(database_url, pool_pre_ping=True)

In [5]:
# 연결 확인
pd.read_sql(text('SELECT 1 AS ok'), engine)

,ok
0,1


In [ ]:
# 조회 쿼리만 여기에 작성하세요.
# 000bef200b724f6b8876 사용자의 AT_USER_SCALE_SCORE 테이블 조회 예시
query = text('''
SELECT
    *
FROM AT_USER_SCALE_SCORE
WHERE USER_TESTING_NO = '000bef200b724f6b8876';
''')

df = pd.read_sql(query, engine)
df

,USER_TESTING_NO,SCALE_CODE,YYYYMM,RAW_TOTAL_SCORE,ORIGIN_TOTAL_SCORE,SUBS_YN
0,000bef200b724f6b8876,A08,201512,60,60,None
1,000bef200b724f6b8876,B08,201512,70,70,None
2,000bef200b724f6b8876,C08,201512,160,160,None
3,000bef200b724f6b8876,D08,201512,70,70,None
4,000bef200b724f6b8876,E08,201512,90,90,None
5,000bef200b724f6b8876,F08,201512,60,60,None
6,000bef200b724f6b8876,Z08,201512,510,510,None


In [20]:
# 조회 쿼리만 여기에 작성하세요.
# 000bef200b724f6b8876 사용자의 응답 결과 조회 예시
query = text('''
SELECT
    *
FROM AT_USER_TESTING_PAPER
WHERE USER_TESTING_NO = '0003345871af48aabf56';
''')

df = pd.read_sql(query, engine)
df

,USER_TESTING_NO,PAPER_JSON,REG_DATE
0,0003345871af48aabf56,"{""questionChoiceList"":[{""questionNo"":""1"",""choi...",2018-06-11 11:47:02


In [ ]:
# Preprocess JSON: questionNo columns, choiceNo/choiceScore rows, then save to Excel
import json
from pathlib import Path

rows = []

for _, row in df.iterrows():
    paper_json = json.loads(row['PAPER_JSON'])
    questions = pd.DataFrame(paper_json['questionChoiceList']) # 데이터 프레임 화
    questions['questionNo'] = questions['questionNo'].astype(str) # questionNo를 문자열로 속성 변환

    wide = questions.set_index('questionNo')[['choiceNo', 'choiceScore']].T.reset_index()
    wide = wide.rename(columns={'index': 'value_type'})
    wide.insert(0, 'REG_DATE', row.get('REG_DATE'))
    wide.insert(0, 'USER_TESTING_NO', row.get('USER_TESTING_NO'))
    rows.append(wide)

response_excel = pd.concat(rows, ignore_index=True)
response_excel.columns.name = None

excel_path = Path('paper_json_preprocessed.xlsx')
response_excel.to_excel(excel_path, index=False)

print(f'Saved: {excel_path.resolve()}')
display(response_excel)